# Visualize how MFA/HDDC Gaussians tile one toy manifold

Give this notebook a completed toy-manifold run directory. It selects one planted manifold instance, fits PCA only to that manifold's points, and projects both the points and the fitted Gaussian components into the same 2D or 3D space.

PCA is intentional here: a linear projection maps a Gaussian to another Gaussian exactly, so the plotted ellipses/ellipsoids are the true projected covariances $P\Sigma_kP^\top$, with $\Sigma_k=W_kW_k^\top+\Psi_k$. Nonlinear methods such as t-SNE or UMAP would not preserve a Gaussian covariance.

Choose `ASSIGNMENT_MODE = "responsibility"` for the saved MFA responsibility argmax, or `"nearest_mean"` for the nearest learned $\mu_k$ in Euclidean distance. Component matching and point colors follow the selected rule. Nearest-mean assignments are computed in memory and never overwrite the saved assignment bundle.

In [31]:
from pathlib import Path

# Point this at any completed single-file toy-manifold MFA/HDDC run.
RUN_DIR = Path(
    "dalg-cache/toy_manifold_models_20k/"
    "adaptive_q_toy_20k_hddc_shared_b_surgery_point3/"
    "hddc__toy_manifolds_d128_20k_noise1e4__l00__k200__q16__s42__81375355"
)

MANIFOLD_ID = 1             # inspect manifold_metadata['manifolds'] below
PLOT_DIM = 3                # 2 for the circle; use 3 for helix/surface geometry
N_STD = 1.0                 # radius of optional Gaussian footprints
ASSIGNMENT_MODE = "nearest_mean"  # "responsibility" or "nearest_mean"
COMPONENT_SELECTION = "dominant"  # "dominant" or "touching"
SHOW_GAUSSIAN_FOOTPRINTS = False
SHOW_COVARIANCE_PC_AXES = True
COVARIANCE_PC_COUNT = 1     # show PC1 only by default
COVARIANCE_AXIS_SCALE = 0.45  # half-length in covariance standard deviations
COVARIANCE_AXIS_OPACITY = 0.35
COVARIANCE_AXIS_WIDTH = 2
MIN_POINTS_PER_COMPONENT = 1
MAX_PLOT_POINTS = 10_000
POINT_SIZE = 6
POINT_OPACITY = 0.9
GAUSSIAN_OPACITY = 0.10
ANNOTATE_COMPONENT_IDS = False
RANDOM_SEED = 0

## Load and validate the run

The notebook streams all points once in canonical row order. In `responsibility` mode it validates and uses the run's existing `mfa_model_assignments.pt`; in `nearest_mean` mode it assigns every point to the closest learned component mean using Euclidean distance. It never writes an assignment artifact.

In [32]:
import json
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

REPO = Path.cwd().resolve()
if not (REPO / "src").is_dir():
    REPO = REPO.parent.resolve()
if not (REPO / "src").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks/.")
sys.path.insert(0, str(REPO / "src"))

from dalg.analysis.nearest_centroid_assignments import compute_nearest_centroid_assignments
from dalg.data.shard_activations import ActivationBatchDataset, load_meta_index

RUN_DIR = RUN_DIR.expanduser()
if not RUN_DIR.is_absolute():
    RUN_DIR = REPO / RUN_DIR
MODEL_PATH = RUN_DIR / "mfa_model.pt"
ASSIGNMENTS_PATH = RUN_DIR / "mfa_model_assignments.pt"
required = [RUN_DIR / "config.json", MODEL_PATH]
if ASSIGNMENT_MODE == "responsibility":
    required.append(ASSIGNMENTS_PATH)
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing run artifacts: {missing}")
if PLOT_DIM not in (2, 3):
    raise ValueError("PLOT_DIM must be 2 or 3.")
if not 1 <= COVARIANCE_PC_COUNT <= PLOT_DIM:
    raise ValueError("COVARIANCE_PC_COUNT must lie between 1 and PLOT_DIM.")
if COVARIANCE_AXIS_SCALE <= 0:
    raise ValueError("COVARIANCE_AXIS_SCALE must be positive.")
if not 0 <= COVARIANCE_AXIS_OPACITY <= 1:
    raise ValueError("COVARIANCE_AXIS_OPACITY must lie in [0, 1].")
if ASSIGNMENT_MODE not in {"responsibility", "nearest_mean"}:
    raise ValueError("ASSIGNMENT_MODE must be 'responsibility' or 'nearest_mean'.")
if COMPONENT_SELECTION not in {"dominant", "touching"}:
    raise ValueError("COMPONENT_SELECTION must be 'dominant' or 'touching'.")

run_config = json.loads((RUN_DIR / "config.json").read_text())
model_kind = run_config.get("model")
if model_kind == "MFA_HDDC":
    from dalg.models.adaptive_q.mfa_hddc import load_mfa_hddc
    model = load_mfa_hddc(MODEL_PATH, map_location="cpu")
elif model_kind == "MFA_ARD":
    from dalg.models.adaptive_q.mfa_ard import load_mfa_ard
    model = load_mfa_ard(MODEL_PATH, map_location="cpu")
elif model_kind == "MFA":
    from dalg.models.mfa import load_mfa
    model = load_mfa(MODEL_PATH, map_location="cpu")
else:
    raise ValueError(f"Unsupported model kind in config.json: {model_kind!r}")
model.eval()

shard_dir = Path(run_config["shard_dir"]).expanduser()
if not shard_dir.is_absolute():
    shard_dir = REPO / shard_dir
shard_config = json.loads((shard_dir / "config.json").read_text())
if shard_config.get("source_kind") != "toy_manifolds":
    raise ValueError("This notebook requires toy-manifold shards with planted labels.")
if int(shard_config["window"]) != 1 or int(shard_config.get("drop_prefix", 0)) != 0:
    raise ValueError("Expected exactly one activation per toy-manifold row.")

metadata_path = shard_dir / shard_config["manifold_metadata"]
manifold_metadata = torch.load(metadata_path, map_location="cpu", weights_only=True)
row_manifold_ids = manifold_metadata["row_manifold_ids"].reshape(-1).long()
num_manifolds = int(manifold_metadata["num_manifolds"])
if not 0 <= MANIFOLD_ID < num_manifolds:
    raise ValueError(f"MANIFOLD_ID must be in [0, {num_manifolds - 1}].")

layer = int(run_config["layer"])
meta_index = load_meta_index(shard_dir, layer=layer)
if len(meta_index) != row_manifold_ids.numel():
    raise ValueError("Shard rows and planted manifold labels are not aligned.")

all_positions = list(range(len(meta_index)))
dataset = ActivationBatchDataset(
    shard_dir, layer=layer, row_subset=all_positions, batch_size=4096,
    drop_prefix=0, dtype=torch.float32, shuffle_shards=False,
    shuffle_within_shard=False, seed=0,
)
all_points = torch.cat(
    list(DataLoader(dataset, batch_size=None, num_workers=0)), dim=0
).cpu()
if all_points.shape != (len(meta_index), model.D):
    raise ValueError(
        f"Expected canonical point matrix {(len(meta_index), model.D)}, "
        f"got {tuple(all_points.shape)}."
    )

if ASSIGNMENT_MODE == "responsibility":
    assignment_bundle = torch.load(
        ASSIGNMENTS_PATH, map_location="cpu", mmap=True, weights_only=True
    )
    assignments = assignment_bundle["assignments"].reshape(-1).long()
    cluster_sizes = assignment_bundle["cluster_sizes"].reshape(-1).long()
    if int(assignment_bundle["K"]) != model.K:
        raise ValueError("Assignment K does not match the model.")
else:
    cluster_sizes, assignments, _ = compute_nearest_centroid_assignments(
        model.mu.detach().cpu(), all_points, device="cpu", batch_size=4096
    )
if assignments.numel() != len(meta_index) or cluster_sizes.numel() != model.K:
    raise ValueError("Assignments do not match the canonical dataset or model K.")
if not torch.equal(torch.bincount(assignments, minlength=model.K), cluster_sizes):
    raise ValueError("cluster_sizes is inconsistent with assignments.")

manifold_positions = torch.where(row_manifold_ids == MANIFOLD_ID)[0]
manifold_points = all_points[manifold_positions]
manifold_assignments = assignments[manifold_positions]
if manifold_points.shape[0] != manifold_assignments.numel():
    raise ValueError("Selected activations and assignments lost alignment.")

manifold_info = manifold_metadata["manifolds"][MANIFOLD_ID]
display(pd.DataFrame(manifold_metadata["manifolds"])[
    ["manifold_id", "type_name", "intrinsic_dim", "embedding_dim", "noise_std"]
])
print(
    f"Loaded {model_kind}: K={model.K}, D={model.D}, q_max={model.q}. "
    f"Selected manifold {MANIFOLD_ID} ({manifold_info['type_name']}) with "
    f"{manifold_points.shape[0]:,} points using {ASSIGNMENT_MODE} assignments."
)

nearest-centroid assignments: 100%|██████████| 5/5 [00:00<00:00, 414.85it/s]


,manifold_id,type_name,intrinsic_dim,embedding_dim,noise_std
0,0,circle,1,2,"tensor(0.0001, dtype=torch.float64)"
1,1,helix,1,3,"tensor(8.4215e-05, dtype=torch.float64)"


Loaded MFA_HDDC: K=200, D=128, q_max=16. Selected manifold 1 (helix) with 10,000 points using nearest_mean assignments.


## Match Gaussian components to the manifold

The table follows the active assignment rule. `purity` is the fraction of all points assigned to a component that come from this manifold.

In [33]:
component_by_manifold = torch.bincount(
    assignments * num_manifolds + row_manifold_ids,
    minlength=model.K * num_manifolds,
).reshape(model.K, num_manifolds)
dominant_manifold = component_by_manifold.argmax(dim=1)
points_here = component_by_manifold[:, MANIFOLD_ID]

if COMPONENT_SELECTION == "dominant":
    selected_mask = (dominant_manifold == MANIFOLD_ID) & (cluster_sizes > 0)
else:
    selected_mask = points_here > 0
selected_mask &= points_here >= MIN_POINTS_PER_COMPONENT
selected_components = torch.where(selected_mask)[0]
if selected_components.numel() == 0:
    raise ValueError("No components meet the current selection settings.")

if hasattr(model, "component_ranks"):
    component_ranks = model.component_ranks.detach().cpu().long()
elif hasattr(model, "effective_ranks"):
    component_ranks = model.effective_ranks().detach().cpu().long()
else:
    component_ranks = torch.full((model.K,), model.q, dtype=torch.long)
mixture_weights = model.pi_logits.softmax(dim=0).detach().cpu()
purity = points_here.float() / cluster_sizes.float().clamp_min(1)
component_table = pd.DataFrame({
    "component": selected_components.numpy(),
    "points_on_manifold": points_here[selected_components].numpy(),
    "total_assigned_points": cluster_sizes[selected_components].numpy(),
    "purity": purity[selected_components].numpy(),
    "learned_rank": component_ranks[selected_components].numpy(),
    "mixture_weight": mixture_weights[selected_components].numpy(),
}).sort_values("component").reset_index(drop=True)
print(
    f"Selected {len(component_table)} components; they receive "
    f"{int(points_here[selected_components].sum()):,} / {len(manifold_positions):,} "
    "points from this manifold."
)
display(component_table)

Selected 86 components; they receive 10,000 / 10,000 points from this manifold.


,component,points_on_manifold,total_assigned_points,purity,learned_rank,mixture_weight
0,0,149,149,1.0,2,0.006110
1,2,38,38,1.0,2,0.004741
2,3,171,171,1.0,1,0.004983
3,4,97,97,1.0,16,0.004090
4,7,260,260,1.0,1,0.005578
...,...,...,...,...,...,...
81,184,53,53,1.0,1,0.004953
82,190,156,156,1.0,3,0.004842
83,191,61,61,1.0,1,0.005487
84,193,245,245,1.0,1,0.005617


## Fit PCA and project the fitted Gaussians

PCA is fitted on all points from the selected manifold, not on the component means. The covariance projection includes the model's learned diagonal noise, including the shared scalar $b$ in shared-$b$ HDDC runs.

In [34]:
from sklearn.decomposition import PCA

pca = PCA(n_components=PLOT_DIM, svd_solver="full")
points_projected = pca.fit_transform(manifold_points.numpy())
basis = torch.from_numpy(pca.components_).to(model.mu.dtype)  # (p, D)
pca_center = torch.from_numpy(pca.mean_).to(model.mu.dtype)

with torch.no_grad():
    means = model.mu.detach().cpu()
    loadings = model._W().detach().cpu()
    psi = model._psi().detach().cpu()
    means_projected = (means - pca_center) @ basis.T
    loadings_projected = torch.einsum("pd,kdq->kpq", basis, loadings)
    loading_covariance = torch.einsum(
        "kpq,krq->kpr", loadings_projected, loadings_projected
    )
    noise_covariance = torch.einsum("pd,kd,rd->kpr", basis, psi, basis)
    covariance_projected = loading_covariance + noise_covariance

means_projected = means_projected.numpy()
covariance_projected = covariance_projected.numpy()
explained = pca.explained_variance_ratio_
display(pd.DataFrame({
    "PCA axis": np.arange(1, PLOT_DIM + 1),
    "explained variance": explained,
    "cumulative": np.cumsum(explained),
}))

,PCA axis,explained variance,cumulative
0,1,0.474902,0.474902
1,2,0.324321,0.799223
2,3,0.200776,0.999999


## Tiling plot

Points and Gaussian footprints use the same component color. Gaussian footprints are borderless translucent regions rather than outline or wireframe meshes; the larger, nearly opaque points are rendered afterward so the dataset remains the visual focus. Hover over points or mean markers to inspect component IDs.

In [35]:
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

selected_ids = selected_components.tolist()
palette = sample_colorscale("Turbo", np.linspace(0.03, 0.97, len(selected_ids)))
component_color = {component_id: palette[i] for i, component_id in enumerate(selected_ids)}
rng = np.random.default_rng(RANDOM_SEED)
if len(points_projected) > MAX_PLOT_POINTS:
    plot_indices = np.sort(rng.choice(len(points_projected), MAX_PLOT_POINTS, replace=False))
else:
    plot_indices = np.arange(len(points_projected))

def point_limits(values, margin_fraction=0.08):
    low = values.min(axis=0)
    high = values.max(axis=0)
    margin = np.maximum((high - low) * margin_fraction, 1e-6)
    return low - margin, high + margin

def with_alpha(rgb, alpha):
    return rgb.replace("rgb(", "rgba(").replace(")", f", {alpha})")

def ellipse_polygon(mean, covariance, n_std, num_points=100):
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    transform = eigenvectors @ np.diag(n_std * np.sqrt(np.maximum(eigenvalues, 0.0)))
    angle = np.linspace(0.0, 2.0 * np.pi, num_points)
    unit_circle = np.stack([np.cos(angle), np.sin(angle)])
    return mean[:, None] + transform @ unit_circle

def ellipsoid_mesh(mean, covariance, n_std, num_u=24, num_v=13):
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    transform = eigenvectors @ np.diag(n_std * np.sqrt(np.maximum(eigenvalues, 0.0)))
    u = np.linspace(0.0, 2.0 * np.pi, num_u, endpoint=False)
    v = np.linspace(0.0, np.pi, num_v)
    uu, vv = np.meshgrid(u, v)
    sphere = np.stack([
        np.cos(uu) * np.sin(vv),
        np.sin(uu) * np.sin(vv),
        np.cos(vv),
    ]).reshape(3, -1)
    vertices = mean[:, None] + transform @ sphere
    triangles_i, triangles_j, triangles_k = [], [], []
    for row in range(num_v - 1):
        for col in range(num_u):
            next_col = (col + 1) % num_u
            a = row * num_u + col
            b = row * num_u + next_col
            c = (row + 1) * num_u + col
            d = (row + 1) * num_u + next_col
            triangles_i.extend([a, b])
            triangles_j.extend([b, d])
            triangles_k.extend([c, c])
    return vertices, triangles_i, triangles_j, triangles_k

shown_assignments = manifold_assignments.numpy()[plot_indices]
shown_points = points_projected[plot_indices]
selected_point_mask = np.isin(shown_assignments, selected_ids)
fig = go.Figure()

# Build covariance overlays around the projected model means.
pc_axis_segments = []
for component_id in selected_ids:
    color = component_color[component_id]
    mean = means_projected[component_id]
    covariance = covariance_projected[component_id]
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    for pc_offset in range(COVARIANCE_PC_COUNT):
        eigen_index = PLOT_DIM - 1 - pc_offset
        half_axis = (
            COVARIANCE_AXIS_SCALE * np.sqrt(max(eigenvalues[eigen_index], 0.0))
            * eigenvectors[:, eigen_index]
        )
        pc_axis_segments.append((
            component_id, pc_offset + 1, mean - half_axis, mean + half_axis
        ))
    if SHOW_GAUSSIAN_FOOTPRINTS and PLOT_DIM == 2:
        polygon = ellipse_polygon(mean, covariance, N_STD)
        fig.add_trace(go.Scatter(
            x=polygon[0], y=polygon[1], mode="lines",
            line=dict(width=0, color=with_alpha(color, 0.0)),
            fill="toself", fillcolor=with_alpha(color, GAUSSIAN_OPACITY),
            hoverinfo="skip", showlegend=False,
        ))
    elif SHOW_GAUSSIAN_FOOTPRINTS:
        vertices, face_i, face_j, face_k = ellipsoid_mesh(mean, covariance, N_STD)
        fig.add_trace(go.Mesh3d(
            x=vertices[0], y=vertices[1], z=vertices[2],
            i=face_i, j=face_j, k=face_k, color=color,
            opacity=GAUSSIAN_OPACITY, flatshading=False,
            hoverinfo="skip", showscale=False, showlegend=False,
        ))

# Dataset points are added after the footprints and remain visually dominant.
point_trace = go.Scattergl if PLOT_DIM == 2 else go.Scatter3d
if (~selected_point_mask).any():
    gray_points = shown_points[~selected_point_mask]
    coordinates = dict(x=gray_points[:, 0], y=gray_points[:, 1])
    if PLOT_DIM == 3:
        coordinates["z"] = gray_points[:, 2]
    fig.add_trace(point_trace(
        **coordinates, mode="markers",
        marker=dict(size=POINT_SIZE, color="rgb(135, 135, 135)", opacity=0.65),
        hoverinfo="skip", showlegend=False,
    ))

visible_points = shown_points[selected_point_mask]
visible_assignments = shown_assignments[selected_point_mask]
visible_colors = [component_color[int(component_id)] for component_id in visible_assignments]
coordinates = dict(x=visible_points[:, 0], y=visible_points[:, 1])
if PLOT_DIM == 3:
    coordinates["z"] = visible_points[:, 2]
point_hover = (
    "component %{customdata}<br>PC1 %{x:.4f}<br>PC2 %{y:.4f}"
    + ("<br>PC3 %{z:.4f}" if PLOT_DIM == 3 else "")
    + "<extra></extra>"
)
fig.add_trace(point_trace(
    **coordinates, mode="markers", customdata=visible_assignments,
    marker=dict(size=POINT_SIZE, color=visible_colors, opacity=POINT_OPACITY),
    hovertemplate=point_hover, showlegend=False,
))

# Each segment is centered exactly at mu_k and spans mu_k ± n_std sqrt(lambda) v.
if SHOW_COVARIANCE_PC_AXES:
    axis_trace = go.Scatter if PLOT_DIM == 2 else go.Scatter3d
    for component_id, pc_number, start, end in pc_axis_segments:
        coordinates = dict(x=[start[0], end[0]], y=[start[1], end[1]])
        if PLOT_DIM == 3:
            coordinates["z"] = [start[2], end[2]]
        fig.add_trace(axis_trace(
            **coordinates, mode="lines",
            line=dict(
                color=with_alpha(component_color[component_id], COVARIANCE_AXIS_OPACITY),
                width=max(1, COVARIANCE_AXIS_WIDTH - pc_number + 1),
            ),
            hovertemplate=(
                f"component {component_id}<br>covariance PC{pc_number}"
                f"<br>centered at learned mean<extra></extra>"
            ),
            showlegend=False,
        ))

selected_means = means_projected[selected_ids]
mean_trace = go.Scattergl if PLOT_DIM == 2 else go.Scatter3d
mean_coordinates = dict(x=selected_means[:, 0], y=selected_means[:, 1])
if PLOT_DIM == 3:
    mean_coordinates["z"] = selected_means[:, 2]
mean_mode = "markers+text" if ANNOTATE_COMPONENT_IDS else "markers"
mean_hover = (
    "mean of component %{customdata}<br>PC1 %{x:.4f}<br>PC2 %{y:.4f}"
    + ("<br>PC3 %{z:.4f}" if PLOT_DIM == 3 else "")
    + "<extra></extra>"
)
fig.add_trace(mean_trace(
    **mean_coordinates, mode=mean_mode, customdata=selected_ids,
    text=[str(component_id) for component_id in selected_ids] if ANNOTATE_COMPONENT_IDS else None,
    textposition="top center",
    marker=dict(size=9, symbol="x", color=[component_color[i] for i in selected_ids]),
    hovertemplate=mean_hover, showlegend=False,
))

low, high = point_limits(points_projected)
overlay_labels = []
if SHOW_COVARIANCE_PC_AXES:
    pc_label = "PC1" if COVARIANCE_PC_COUNT == 1 else f"PC1–PC{COVARIANCE_PC_COUNT}"
    overlay_labels.append(f"centered covariance {pc_label}")
if SHOW_GAUSSIAN_FOOTPRINTS:
    overlay_labels.append(f"{N_STD:g}σ Gaussian footprints")
overlay_label = " + ".join(overlay_labels) if overlay_labels else "centroids only"
title = (
    f"{RUN_DIR.name}<br>"
    f"manifold {MANIFOLD_ID}: {manifold_info['type_name']} | "
    f"{ASSIGNMENT_MODE} assignments | {len(selected_ids)} components | {overlay_label}"
)
fig.update_layout(
    template="plotly_white", width=1000, height=850, title=title,
    margin=dict(l=55, r=35, t=95, b=55), showlegend=False,
)
if PLOT_DIM == 2:
    fig.update_xaxes(
        range=[low[0], high[0]], title=f"PC1 ({explained[0]:.1%} variance)",
        zeroline=False, gridcolor="rgba(0, 0, 0, 0.08)",
    )
    fig.update_yaxes(
        range=[low[1], high[1]], title=f"PC2 ({explained[1]:.1%} variance)",
        scaleanchor="x", scaleratio=1, zeroline=False,
        gridcolor="rgba(0, 0, 0, 0.08)",
    )
else:
    fig.update_layout(scene=dict(
        xaxis=dict(range=[low[0], high[0]], title=f"PC1 ({explained[0]:.1%})"),
        yaxis=dict(range=[low[1], high[1]], title=f"PC2 ({explained[1]:.1%})"),
        zaxis=dict(range=[low[2], high[2]], title=f"PC3 ({explained[2]:.1%})"),
        aspectmode="data", bgcolor="white",
    ))
fig.show(config=dict(scrollZoom=True, displaylogo=False, responsive=True))

## Reading the plot

A successful local tiling should show clearly visible component-colored point segments arranged along the manifold. The same-colored PC1 segment is the leading eigenvector of the projected covariance, scaled by its standard deviation and centered exactly on the component's learned mean (the cross). It should align with the local manifold tangent. Means displaced from their colored points indicate that optimization and hard assignments disagree locally.

Set `COVARIANCE_PC_COUNT = 2` to include the second covariance axis, or `SHOW_GAUSSIAN_FOOTPRINTS = True` to restore the marginal confidence regions. Drag to pan or rotate, scroll to zoom, and hover for component IDs.